# Relaciones entre clases en UML

Bienvenido/a. En esta lección vas a distinguir los 6 tipos de relación entre clases que existen en UML, de la más débil a la más fuerte, y a reconocerlos en código real.

## Objetivos
- Distinguir dependencia, asociación, agregación, composición, herencia y realización.
- Reconocer cada relación mirando el código Python (qué se guarda como atributo, qué solo se recibe como parámetro, y qué se hereda o implementa).
- Saber cuándo una relación es débil (la parte sobrevive al todo) o fuerte (la parte muere con el todo), y distinguir "tiene-un" (composición/agregación) de "es-un" (herencia/realización).

---

**Ejemplo de la vida real:** piensa en cuánto depende una cosa de otra para existir:
- Pides un taxi (lo *usas* un momento: dependencia).
- Tienes un teléfono guardado en tu directorio (lo *tienes* de forma permanente, pero existe con independencia de ti: asociación).
- Un equipo de fútbol tiene jugadores, pero si el equipo se disuelve, los jugadores siguen jugando en otro lado (agregación).
- Tu corazón es parte de ti: no existe fuera de tu cuerpo (composición).
- Un estudiante y un profesor son-un tipo de persona: ambos comparten nombre y saber saludar, sin repetir ese código (herencia).
- Un estudiante y un profesor pueden ser notificados: ambos cumplen el mismo contrato ("puedo recibir un mensaje"), pero cada uno lo hace a su manera (realización).

## Explicación detallada

| Relación | Fuerza | Cómo se ve en el código | Notación UML |
|---|---|---|---|
| Dependencia | La más débil | La clase aparece solo como **parámetro de un método**, nunca como atributo | Flecha punteada `- - >` |
| Asociación | Media | La clase se guarda como **atributo** en `__init__`, de forma permanente | Flecha continua `—>` |
| Agregación | Fuerte, pero la parte sobrevive | El objeto "parte" **ya existía** antes y se agrega a una colección del "todo" | Rombo vacío `◇—` del lado del todo |
| Composición | La más fuerte (tiene-un) | El objeto "parte" se **crea dentro de** `__init__` del "todo", nadie más lo referencia | Rombo lleno `◆—` del lado del todo |
| Herencia | Es-un — comparte implementación | La subclase se declara `class Hija(Padre):` y reutiliza atributos/métodos del padre | Flecha continua, punta triangular vacía `--\|>` |
| Realización | Es-un — comparte solo el contrato | La clase implementa los métodos de una interfaz/clase abstracta (`ABC` + `@abstractmethod`) | Flecha punteada, punta triangular vacía `..\|>` |

La pregunta clave para distinguir agregación de composición: **si destruyo el "todo", ¿la "parte" sigue teniendo sentido por sí sola?** Si sí (el jugador sigue jugando en otro equipo) es agregación. Si no (el motor de ese carro específico ya no sirve para nada) es composición.

La pregunta clave para distinguir herencia de realización: **¿la clase padre le da código ya escrito, o solo le exige una firma de método?** Si Persona le regala `saludar()` ya implementado a Estudiante, es herencia. Si Notificable solo obliga a que exista `notificar()`, sin decir cómo, es realización.

Las cuatro primeras relaciones (dependencia → composición) describen **qué tiene** una clase (relaciones "tiene-un" / *has-a*). Las dos últimas (herencia y realización) describen **qué es** una clase (relaciones "es-un" / *is-a*).

## Ejemplo práctico 1 — Dependencia y Asociación

Dominio `Course`-`Student`-`Professor` (el mismo de `ejemplo_uml.py`).

In [1]:
class Course:
    def get_knowledge(self) -> str:
        return "knowledge"

class Student:
    def remember(self, knowledge: str) -> None:
        print(f"El estudiante ha recordado {knowledge}")

class Professor:
    def __init__(self, student: Student) -> None:
        # 'student' se guarda como atributo -> ASOCIACIÓN (permanente)
        self.student = student

    def teach(self, course: Course) -> None:
        # 'course' solo llega como parámetro, no se guarda -> DEPENDENCIA (momentanea)
        self.student.remember(course.get_knowledge())

student = Student()
course = Course()
professor = Professor(student)
professor.teach(course)
print("¿Professor guarda a Student?", hasattr(professor, "student"))
print("¿Professor guarda a Course?", hasattr(professor, "course"))

El estudiante ha recordado knowledge
¿Professor guarda a Student? True
¿Professor guarda a Course? False


## Ejemplo práctico 2 — Agregación (Equipo — Jugador)

El `Jugador` existe **antes** de ser fichado, y sigue existiendo si el equipo se disuelve.

In [2]:
class Jugador:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre

class Equipo:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.jugadores: list[Jugador] = []  # AGREGACIÓN: guarda referencias, no crea jugadores

    def fichar(self, jugador: Jugador) -> None:
        self.jugadores.append(jugador)

    def disolver(self) -> None:
        self.jugadores = []  # los jugadores NO desaparecen, solo dejan de estar aquí

equipo = Equipo("Tiburones")
ana = Jugador("Ana")
luis = Jugador("Luis")
equipo.fichar(ana)
equipo.fichar(luis)
print(f"{equipo.nombre} tiene a {[j.nombre for j in equipo.jugadores]}")
equipo.disolver()
print(f"Equipo disuelto. ¿Ana sigue existiendo?: {ana.nombre}")

Tiburones tiene a ['Ana', 'Luis']
Equipo disuelto. ¿Ana sigue existiendo?: Ana


## Ejemplo práctico 3 — Composición (Carro — Motor)

El `Motor` se crea **dentro de** `Carro.__init__` — nadie más tiene esa instancia.

In [3]:
class Motor:
    def __init__(self, caballos_de_fuerza: int) -> None:
        self.caballos_de_fuerza = caballos_de_fuerza

class Carro:
    def __init__(self, modelo: str, caballos_de_fuerza: int) -> None:
        self.modelo = modelo
        self.motor = Motor(caballos_de_fuerza)  # COMPOSICIÓN: se crea aquí, no se recibe de afuera

    def destruir(self) -> None:
        self.motor = None  # ESTE motor deja de existir junto con el carro

carro = Carro("Corolla", 140)
print(f"{carro.modelo} tiene un motor de {carro.motor.caballos_de_fuerza} hp")
carro.destruir()
print(f"Carro destruido. ¿Motor accesible?: {carro.motor}")

Corolla tiene un motor de 140 hp
Carro destruido. ¿Motor accesible?: None


## Ejemplo práctico 4 — Herencia (Persona → Estudiante, Profesor)

`Estudiante` y `Profesor` **son-un** `Persona`: ambos comparten `nombre` y `saludar()`, definidos una sola vez en la clase padre.

In [4]:
class Persona:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre

    def saludar(self) -> str:
        return f"Hola, soy {self.nombre}"

class Estudiante(Persona):  # HERENCIA: Estudiante ES-UN Persona
    def remember(self, knowledge: str) -> None:
        print(f"{self.nombre} ha recordado {knowledge}")

class Profesor(Persona):  # HERENCIA: Profesor ES-UN Persona
    def __init__(self, nombre: str, estudiante: Estudiante) -> None:
        super().__init__(nombre)  # reutiliza el __init__ de Persona
        self.estudiante = estudiante  # esto además es una ASOCIACIÓN

    def teach(self, knowledge: str) -> None:
        self.estudiante.remember(knowledge)

ana = Estudiante("Ana")
juan = Profesor("Juan", ana)
print(ana.saludar())    # heredado de Persona, no reescrito en Estudiante
print(juan.saludar())   # heredado de Persona, no reescrito en Profesor
juan.teach("polimorfismo")
print("¿Estudiante es-un Persona?", isinstance(ana, Persona))
print("¿Profesor es-un Persona?", issubclass(Profesor, Persona))

Hola, soy Ana
Hola, soy Juan
Ana ha recordado polimorfismo
¿Estudiante es-un Persona? True
¿Profesor es-un Persona? True


## Ejemplo práctico 5 — Realización (interfaz Notificable)

A diferencia de la herencia, aquí `Notificable` no regala código: solo exige que exista `notificar()`. Cada clase **cumple el contrato** a su manera — eso es realización.

In [5]:
from abc import ABC, abstractmethod

class Notificable(ABC):  # interfaz: no implementa nada, solo exige la firma
    @abstractmethod
    def notificar(self, mensaje: str) -> None: ...

class EstudianteNotificable(Estudiante, Notificable):  # REALIZACIÓN
    def notificar(self, mensaje: str) -> None:
        print(f"[Estudiante {self.nombre}] {mensaje}")

class ProfesorNotificable(Profesor, Notificable):  # REALIZACIÓN
    def notificar(self, mensaje: str) -> None:
        print(f"[Profesor {self.nombre}] {mensaje}")

ana2 = EstudianteNotificable("Ana")
juan2 = ProfesorNotificable("Juan", ana2)
ana2.notificar("Tu tarea vence mañana")
juan2.notificar("Tienes una reunión a las 3pm")

# Si a EstudianteNotificable le faltara notificar(), Python ni siquiera
# dejaría instanciarla: el contrato de Notificable es obligatorio.
try:
    class Incompleto(Notificable):
        pass
    Incompleto()
except TypeError as e:
    print(f"Error esperado: {e}")

[Estudiante Ana] Tu tarea vence mañana
[Profesor Juan] Tienes una reunión a las 3pm
Error esperado: Can't instantiate abstract class Incompleto without an implementation for abstract method 'notificar'


## Ejercicios prácticos y preguntas de reflexión

1. Agrega una clase `Entrenador` que sea **asociación** de `Equipo` (el equipo lo guarda como atributo, pero el entrenador podría dirigir otro equipo después).
2. Agrega una clase `Rueda` que sea **composición** de `Carro` (4 ruedas creadas junto con el carro en `__init__`).
3. Agrega una clase `Administrativo(Persona)` que **herede** de `Persona` y agregue un atributo propio `oficina: str`.
4. Haz que `Administrativo` también **implemente** `Notificable` con su propio `notificar()`.
5. Para cada una de las 6 relaciones de este notebook, escribe en una frase qué pasaría si el "todo" (o la clase padre/interfaz) se destruye o cambia.

### Autoevaluación
- ¿Cómo distingues en código Python una dependencia de una asociación, sin ver ningún diagrama?
- ¿Por qué `self.jugadores: list[Jugador] = []` en `Equipo` es agregación y no composición?
- Da un ejemplo propio (no visto en clase) de composición.
- ¿Qué diferencia hay entre que `Profesor` **herede** de `Persona` y que `Profesor` **implemente** `Notificable`? ¿Cuál te da código gratis y cuál solo te obliga a escribirlo tú?
- Si mañana necesitas que `Course` también pueda ser notificado, ¿lo resolverías con herencia o con realización? ¿Por qué?

## Referencias y recursos
- [Documentación oficial de UML](https://www.uml.org/)
- [Refactoring Guru — UML class diagram relationships](https://refactoring.guru/es/design-patterns/uml)
- [PlantUML: diagramas de clases](https://plantuml.com/es/class-diagram)
- Ver también `class_uml/ejemplo_uml.py` (código fuente completo, ejecutable), `class_uml/uml_class_diagram.puml` (las 6 relaciones en PlantUML) y `class_uml/uml_diagrama_clases.ipynb`.
- Para profundizar en interfaces con `ABC` (clases abstractas, `@abstractmethod`, principio de sustitución de Liskov), ver `class_poo/interfaz.ipynb`.